# Blok NS — Residual U-Net 2D (Denoising)
Skema K1, input deringed **0.8 + 0.4** → target clean **0.2**. Memperbaiki: brightness inference & split apple-to-apple per rot-step.

Arsitektur: Residual U-Net 2D varian restorasi (global residual, MSE+SSIM).

In [ ]:
!pip install -q tifffile scikit-image openpyxl pandas psutil

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports

In [ ]:
import os, gc, time, json, warnings, threading, multiprocessing
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from datetime import datetime
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import GradScaler, autocast

from tifffile import imread, imwrite
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_erosion
import psutil

warnings.filterwarnings('ignore')
CPU_COUNT = multiprocessing.cpu_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print('device:', device, '| CPU cores:', CPU_COUNT)

## 2. Config

In [ ]:
class Config:
    # === BLOK NS (denoising) — input deringed noisy, target clean 0.2 ===
    # Dua rotation step. Target SAMA (clean 0.2) untuk kedua input.
    PATH_CLEAN_02  = "/content/drive/MyDrive/Dataset TA/Libo/rotation degree var/skema K1/Training 512/gtns1_02_crop_z200-712_y200-712_x200-712.tif"
    PATH_IN_08     = "/content/drive/MyDrive/Dataset TA/Libo/rotation degree var/skema K1/Training 512/gtr_08_crop_z200-712_y200-712_x200-712.tif"  # deringed 0.8 (noisy, no ring)
    PATH_IN_04     = "/content/drive/MyDrive/Dataset TA/Libo/rotation degree var/skema K1/Training 512/gtr_04_crop_z200-712_y200-712_x200-712.tif"  # deringed 0.4 (noisy, no ring)

    CROP_SIZE   = None
    CROP_ORIGIN = None
    TRAIN_FRAC  = 0.80
    VAL_FRAC    = 0.10

    PATCH_SIZE        = 256
    PATCHES_PER_SLICE = 20
    STRIDE_TEST       = 96

    BATCH_SIZE  = 16
    EPOCHS      = 100
    LR          = 1e-4
    LR_MIN      = 1e-6
    PATIENCE_ES = 20
    PATIENCE_LR = 8
    LR_FACTOR   = 0.5

    LOSS_TYPE   = 'mse+ssim'
    SSIM_WEIGHT = 0.15
    AUGMENT     = True

    USE_AMP     = False
    NUM_WORKERS = 2

    SAVE_DIR   = "/content/drive/MyDrive/Alfian_TA/K1_2_blokNS_resunet2d"
    MODEL_NAME = 'resunet2d_NS_best.pth'

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
print("BLOK NS | Residual U-Net 2D | save ->", cfg.SAVE_DIR)

## 3. Model: Residual U-Net 2D (restorasi)

In [ ]:
# ============================================================================
#  Residual U-Net 2D — varian RESTORASI (regresi grayscale->grayscale)
#  Adaptasi dari Wang et al. (2024, SPE J) Residual U-Net segmentasi:
#    - output 1 channel (bukan softmax n_classes)
#    - GLOBAL RESIDUAL: out = net(x) + x  (model belajar residual noise/ring)
#    - FILTERS [32,64,128,256,512] -> receptive field besar utk ring artifact
# ============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                                       nn.BatchNorm2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = ResidualBlock(in_ch, out_ch)
        self.pool  = nn.MaxPool2d(2, 2)
    def forward(self, x):
        skip = self.block(x)
        return self.pool(skip), skip

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.block = ResidualBlock(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        return self.block(torch.cat([x, skip], dim=1))

class ResidualUNet2D(nn.Module):
    # Residual U-Net 2D restorasi. in_ch=1. global residual: final = net(x)+x
    FILTERS = [32, 64, 128, 256, 512]
    def __init__(self, in_ch=1, out_ch=1, global_residual=True):
        super().__init__()
        f = self.FILTERS
        self.global_residual = global_residual
        self.enc1 = EncoderBlock(in_ch, f[0]); self.enc2 = EncoderBlock(f[0], f[1])
        self.enc3 = EncoderBlock(f[1], f[2]);  self.enc4 = EncoderBlock(f[2], f[3])
        self.bottleneck = ResidualBlock(f[3], f[4])
        self.dec4 = DecoderBlock(f[4], f[3], f[3]); self.dec3 = DecoderBlock(f[3], f[2], f[2])
        self.dec2 = DecoderBlock(f[2], f[1], f[1]); self.dec1 = DecoderBlock(f[1], f[0], f[0])
        self.out  = nn.Conv2d(f[0], out_ch, kernel_size=1)
    def forward(self, x):
        inp = x
        d, s1 = self.enc1(x); d, s2 = self.enc2(d)
        d, s3 = self.enc3(d); d, s4 = self.enc4(d)
        d = self.bottleneck(d)
        d = self.dec4(d, s4); d = self.dec3(d, s3)
        d = self.dec2(d, s2); d = self.dec1(d, s1)
        r = self.out(d)
        # global residual: prediksi residual lalu tambah input (in_ch=1 → x langsung)
        y = r + inp if self.global_residual else r
        return y

def count_parameters(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 4. Data helpers + Dataset

In [ ]:
def load_and_crop_volume(tif_path, crop_size=None, crop_origin=None):
    print(f"  Loading {os.path.basename(tif_path)} ...", end='', flush=True)
    vol = imread(tif_path).astype(np.float32); print(f" shape={vol.shape}")
    if crop_size is not None:
        D,H,W = vol.shape; cs = crop_size
        if crop_origin is not None: x0,y0,z0 = crop_origin
        else: x0=max(0,(W-cs)//2); y0=max(0,(H-cs)//2); z0=max(0,(D-cs)//2)
        x0=min(x0,W-cs); y0=min(y0,H-cs); z0=min(z0,D-cs)
        vol = vol[z0:z0+cs, y0:y0+cs, x0:x0+cs]
    return vol

def normalize_volume(vol, p1=None, p99=None):
    # Normalisasi adaptif p1/p99; return (vol01, p1, p99) utk denorm balik
    if p1  is None: p1  = np.percentile(vol, 1)
    if p99 is None: p99 = np.percentile(vol, 99)
    vn = np.clip((vol - p1)/(p99 - p1 + 1e-8), 0.0, 1.0)
    return vn.astype(np.float32), float(p1), float(p99)

def split_slices(n, tf, vf):
    nt = int(n*tf); nv = int(n*vf)
    return list(range(0,nt)), list(range(nt,nt+nv)), list(range(nt+nv,n))

class PatchDataset(Dataset):
    def __init__(self, noisy_vol, clean_vol, slice_indices, patch_size,
                 patches_per_slice=20, augment=False, mode='train'):
        self.noisy=noisy_vol; self.clean=clean_vol; self.patch=patch_size
        self.augment=augment; self.mode=mode; self.patches_per_slice=patches_per_slice
        if mode=='train':
            self.slice_indices=slice_indices
            self.length=len(slice_indices)*patches_per_slice
        else:
            stride=max(patch_size//2,32); self.coords=[]; _,H,W=noisy_vol.shape
            for z in slice_indices:
                for y in range(0,H-patch_size+1,stride):
                    for x in range(0,W-patch_size+1,stride):
                        self.coords.append((z,y,x))
            self.length=len(self.coords)
    def __len__(self): return self.length
    def _augment(self,n,c):
        k=np.random.randint(0,4); n=np.rot90(n,k); c=np.rot90(c,k)
        if np.random.rand()>0.5: n=np.fliplr(n); c=np.fliplr(c)
        if np.random.rand()>0.5: n=np.flipud(n); c=np.flipud(c)
        return n.copy(),c.copy()
    def __getitem__(self,idx):
        ps=self.patch; _,H,W=self.noisy.shape
        if self.mode=='train':
            slice_idx=self.slice_indices[idx//self.patches_per_slice]
            y=np.random.randint(0,H-ps); x=np.random.randint(0,W-ps)
        else:
            z,y,x=self.coords[idx]; slice_idx=z
        n_p=self.noisy[slice_idx,y:y+ps,x:x+ps]; c_p=self.clean[slice_idx,y:y+ps,x:x+ps]
        if self.augment and self.mode=='train': n_p,c_p=self._augment(n_p,c_p)
        return torch.from_numpy(n_p[None]).float(), torch.from_numpy(c_p[None]).float()

## 5. Loss (MSE+SSIM)

In [ ]:
class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, C1=0.01**2, C2=0.03**2):
        super().__init__(); self.ws=window_size; self.C1=C1; self.C2=C2
        sigma=1.5
        gauss=torch.Tensor([np.exp(-(x-window_size//2)**2/(2*sigma**2)) for x in range(window_size)])
        gauss/=gauss.sum()
        k=gauss.unsqueeze(1).mm(gauss.unsqueeze(0)).unsqueeze(0).unsqueeze(0)
        self.register_buffer('kernel',k)
    def forward(self,pred,target):
        pred=pred.float(); target=target.float()
        p=self.ws//2; k=self.kernel.float().expand(pred.shape[1],1,-1,-1); Fc=nn.functional.conv2d
        mu_p=Fc(pred,k,padding=p); mu_t=Fc(target,k,padding=p)
        mu_pp=mu_p*mu_p; mu_tt=mu_t*mu_t; mu_pt=mu_p*mu_t
        s_pp=(Fc(pred*pred,k,padding=p)-mu_pp).clamp(min=0.0)
        s_tt=(Fc(target*target,k,padding=p)-mu_tt).clamp(min=0.0)
        s_pt=Fc(pred*target,k,padding=p)-mu_pt
        num=(2*mu_pt+self.C1)*(2*s_pt+self.C2)
        den=(mu_pp+mu_tt+self.C1)*(s_pp+s_tt+self.C2)+1e-8
        ssim=torch.clamp(num/den, -1.0, 1.0)
        return 1.0-ssim.mean()

class CombinedLoss(nn.Module):
    def __init__(self, ssim_weight=0.15):
        super().__init__(); self.mse=nn.MSELoss(); self.ssim=SSIMLoss(); self.w=ssim_weight
    def forward(self,pred,target):
        return (1-self.w)*self.mse(pred,target)+self.w*self.ssim(pred,target)

## 6. Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train(); total=0.0; n_skip=0
    for noisy,clean in loader:
        noisy,clean=noisy.to(device,non_blocking=True),clean.to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=cfg.USE_AMP):
            out=model(noisy); loss=criterion(out,clean)
        if not torch.isfinite(loss):
            n_skip+=1; continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update()
        total+=loss.item()
    if n_skip>0: print(f"    [warn] {n_skip} batch di-skip (loss non-finite)")
    return total/max(1,len(loader)-n_skip)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); total=0.0; n=0
    for noisy,clean in loader:
        noisy,clean=noisy.to(device,non_blocking=True),clean.to(device,non_blocking=True)
        with autocast('cuda', enabled=cfg.USE_AMP):
            l=criterion(model(noisy),clean)
        if torch.isfinite(l): total+=l.item(); n+=1
    return total/max(1,n)

def train_model(cfg, train_loader, val_loader, norm_stats):
    model=ResidualUNet2D(in_ch=1, out_ch=1, global_residual=True).to(device)
    print(f"  Model parameters: {count_parameters(model):,}")
    criterion=(CombinedLoss(cfg.SSIM_WEIGHT) if cfg.LOSS_TYPE=='mse+ssim' else nn.MSELoss()).to(device)
    optimizer=optim.Adam(model.parameters(),lr=cfg.LR)
    scheduler=ReduceLROnPlateau(optimizer,'min',factor=cfg.LR_FACTOR,patience=cfg.PATIENCE_LR,min_lr=cfg.LR_MIN)
    scaler=GradScaler('cuda', enabled=cfg.USE_AMP)
    best_val=float('inf'); pat=0
    history={'train_loss':[],'val_loss':[],'lr':[]}
    model_path=os.path.join(cfg.SAVE_DIR,cfg.MODEL_NAME)
    t_start=time.time()
    for epoch in range(1,cfg.EPOCHS+1):
        t_ep=time.time()
        tr=train_one_epoch(model,train_loader,optimizer,criterion,device,scaler)
        vl=validate(model,val_loader,criterion,device); scheduler.step(vl)
        lr_now=optimizer.param_groups[0]['lr']
        history['train_loss'].append(tr); history['val_loss'].append(vl); history['lr'].append(lr_now)
        if np.isfinite(vl) and vl<best_val:
            best_val=vl; pat=0
            torch.save({'epoch':epoch,'model_state':model.state_dict(),
                'optimizer':optimizer.state_dict(),'val_loss':best_val,
                'arch':'ResidualUNet2D','filters':ResidualUNet2D.FILTERS,
                'config':{k:v for k,v in cfg.__dict__.items() if not k.startswith('__')},
                'norm_stats':norm_stats}, model_path)
            marker=" <- best (disimpan)"
        else:
            pat+=1; marker=f" (patience {pat}/{cfg.PATIENCE_ES})"
        if epoch%5==0 or epoch==1:
            el=(time.time()-t_start)/60
            print(f"  Ep {epoch:4d}/{cfg.EPOCHS}  tr={tr:.5f}  vl={vl:.5f}  "
                  f"lr={lr_now:.2e}  {time.time()-t_ep:.1f}s/ep  elapsed={el:.1f}min{marker}")
        if pat>=cfg.PATIENCE_ES:
            print(f"\n  Early stopping at epoch {epoch}"); break
    print(f"  Best val loss: {best_val:.6f}")
    return model, history, model_path

def plot_training_history(history, save_dir):
    fig,axes=plt.subplots(1,2,figsize=(14,5)); ep=range(1,len(history['train_loss'])+1)
    axes[0].plot(ep,history['train_loss'],label='Train',color='royalblue')
    axes[0].plot(ep,history['val_loss'],label='Val',color='tomato')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].semilogy(ep,history['lr'],color='forestgreen')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('LR'); axes[1].grid(alpha=.3)
    plt.tight_layout(); p=os.path.join(save_dir,'training_history.png')
    plt.savefig(p,dpi=150,bbox_inches='tight'); plt.show(); print("saved",p)

## 7. Inference helpers

In [ ]:
@torch.no_grad()
def infer_slice(model, sl2d, dev, ps=256, stride=96, use_amp=False):
    # Inference 1 slice 2D, patch + Hann blending, output skala [0,1]
    H,W=sl2d.shape; out=np.zeros((H,W),np.float64); wgt=np.zeros((H,W),np.float64)
    g=np.outer(np.hanning(ps),np.hanning(ps))+1e-6
    ys=list(range(0,H-ps+1,stride)); xs=list(range(0,W-ps+1,stride))
    if not ys or ys[-1]+ps<H: ys.append(max(0,H-ps))
    if not xs or xs[-1]+ps<W: xs.append(max(0,W-ps))
    model.eval()
    for y in ys:
        for x in xs:
            t=torch.from_numpy(sl2d[y:y+ps,x:x+ps][None,None]).float().to(dev)
            with autocast('cuda', enabled=use_amp):
                p=model(t).squeeze().float().cpu().numpy()
            out[y:y+ps,x:x+ps]+=p*g; wgt[y:y+ps,x:x+ps]+=g
    return np.clip(out/wgt,0,1).astype(np.float32)

def infer_volume(model, vol01, dev, ps, stride, use_amp, tag='infer'):
    o=np.zeros_like(vol01)
    for z in tqdm(range(vol01.shape[0]), desc=tag):
        o[z]=infer_slice(model, vol01[z], dev, ps, stride, use_amp)
        if z%50==0 and z>0:
            gc.collect()
            if dev.type=='cuda': torch.cuda.empty_cache()
    return o

## 8. Load + normalize (per-volume)

In [ ]:
# ===== LOAD + NORMALISASI per-volume =====
print("="*70); print("  LOAD DATA BLOK NS (input 0.8 & 0.4 -> target clean 0.2)"); print("="*70)

clean02 = load_and_crop_volume(cfg.PATH_CLEAN_02, cfg.CROP_SIZE, cfg.CROP_ORIGIN)
in08    = load_and_crop_volume(cfg.PATH_IN_08,    cfg.CROP_SIZE, cfg.CROP_ORIGIN)
in04    = load_and_crop_volume(cfg.PATH_IN_04,    cfg.CROP_SIZE, cfg.CROP_ORIGIN)

# input pakai stats input; target clean pakai stats clean
cln_n, cln_p1, cln_p99 = normalize_volume(clean02)
in08_n, in08_p1, in08_p99 = normalize_volume(in08)
in04_n, in04_p1, in04_p99 = normalize_volume(in04)

norm_stats = {
    'clean_p1':cln_p1,'clean_p99':cln_p99,
    'in08_p1':in08_p1,'in08_p99':in08_p99,
    'in04_p1':in04_p1,'in04_p99':in04_p99,
}
np.save(os.path.join(cfg.SAVE_DIR,'norm_stats.npy'), norm_stats)

assert cln_n.shape==in08_n.shape==in04_n.shape, "shape harus sama"
n08 = in08_n.shape[0]; n04 = in04_n.shape[0]
print(f"  rot 0.8 in: {in08_n.shape} | rot 0.4 in: {in04_n.shape} | clean target: {cln_n.shape}")

# target gabungan = clean diulang utk tiap input rot-step (sejajar)
# (digabung di sel split)
del clean02, in08, in04; gc.collect()

## 9. Split per-volume (FIX apple-to-apple)

In [ ]:
# ===== SPLIT PER-VOLUME lalu GABUNG (FIX apple-to-apple) =====
tr8, vl8, te8 = split_slices(n08, cfg.TRAIN_FRAC, cfg.VAL_FRAC)
tr4, vl4, te4 = split_slices(n04, cfg.TRAIN_FRAC, cfg.VAL_FRAC)

vol_noisy_all = np.concatenate([in08_n, in04_n], axis=0)
vol_clean_all = np.concatenate([cln_n, cln_n], axis=0)   # clean diulang 2x, sejajar input
rot_label     = np.array(['0.8']*n08 + ['0.4']*n04)

OFF = n08
train_idx = tr8 + [i+OFF for i in tr4]
val_idx   = vl8 + [i+OFF for i in vl4]
test_idx  = te8 + [i+OFF for i in te4]

print(f"  train {len(train_idx)} (0.8:{len(tr8)} | 0.4:{len(tr4)})")
print(f"  val   {len(val_idx)} (0.8:{len(vl8)} | 0.4:{len(vl4)})")
print(f"  test  {len(test_idx)} (0.8:{len(te8)} | 0.4:{len(te4)})  <- seimbang, total {len(test_idx)}")

train_ds=PatchDataset(vol_noisy_all,vol_clean_all,train_idx,cfg.PATCH_SIZE,
                      cfg.PATCHES_PER_SLICE,cfg.AUGMENT,'train')
val_ds  =PatchDataset(vol_noisy_all,vol_clean_all,val_idx,cfg.PATCH_SIZE,
                      cfg.PATCHES_PER_SLICE,False,'val')
lk=dict(batch_size=cfg.BATCH_SIZE,num_workers=cfg.NUM_WORKERS,pin_memory=True)
train_loader=DataLoader(train_ds,shuffle=True,drop_last=True,**lk)
val_loader  =DataLoader(val_ds,shuffle=False,**lk)
print(f"  train patches {len(train_ds):,} | val patches {len(val_ds):,}")

## 10. Train

In [ ]:
# self-test arsitektur
_t=torch.randn(2,1,128,128).to(device); _m=ResidualUNet2D().to(device)
assert _m(_t).shape==_t.shape; print("ResidualUNet2D OK, params:",count_parameters(_m)); del _t,_m
if device.type=='cuda': torch.cuda.empty_cache()

model, history, model_path = train_model(cfg, train_loader, val_loader, norm_stats)
plot_training_history(history, cfg.SAVE_DIR)

## 11. Evaluate test (metrik per rot-step)

In [ ]:
# ===== EVALUASI TEST: metrik terpisah per rot-step + gabungan =====
ckpt=torch.load(model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state']); model.eval()
print(f"  Loaded best: epoch={ckpt['epoch']} val_loss={ckpt['val_loss']:.6f}")

def eval_indices(idx_list):
    rows=[]
    for z in idx_list:
        pred = infer_slice(model, vol_noisy_all[z], device, cfg.PATCH_SIZE, cfg.STRIDE_TEST, cfg.USE_AMP)
        c = vol_clean_all[z]
        rows.append({'slice':z,'rot':rot_label[z],
            'PSNR':psnr_fn(c,pred,data_range=1.0),
            'SSIM':ssim_fn(c,pred,data_range=1.0),
            'RMSE':float(np.sqrt(np.mean((c-pred)**2)))})
    return pd.DataFrame(rows)

df_test = eval_indices(sorted(test_idx))
df_test.to_csv(os.path.join(cfg.SAVE_DIR,'test_metrics_per_slice.csv'), index=False)

print("\n  === METRIK TEST (terpisah per rotation step) ===")
for rot in ['0.8','0.4']:
    sub=df_test[df_test['rot']==rot]
    print(f"  rot {rot}: PSNR {sub['PSNR'].mean():.3f}±{sub['PSNR'].std():.3f}  "
          f"SSIM {sub['SSIM'].mean():.4f}  RMSE {sub['RMSE'].mean():.5f}  (n={len(sub)})")
print(f"  GABUNGAN: PSNR {df_test['PSNR'].mean():.3f}  SSIM {df_test['SSIM'].mean():.4f}  "
      f"RMSE {df_test['RMSE'].mean():.5f}  (n={len(df_test)})")

## 12. Save full-volume (FIX brightness)

In [ ]:
# ===== SIMPAN HASIL FULL-VOLUME (TANPA bug brightness) =====
print("="*70); print("  INFERENCE + SAVE FULL VOLUME (per rot-step terpisah)"); print("="*70)

den08_pred01 = infer_volume(model, in08_n, device, cfg.PATCH_SIZE, cfg.STRIDE_TEST, cfg.USE_AMP, tag="NS 0.8")
den04_pred01 = infer_volume(model, in04_n, device, cfg.PATCH_SIZE, cfg.STRIDE_TEST, cfg.USE_AMP, tag="NS 0.4")

for arr01, name in [(den08_pred01,'NS_denoised_08'), (den04_pred01,'NS_denoised_04')]:
    imwrite(os.path.join(cfg.SAVE_DIR, name+'_float.tif'), arr01.astype(np.float32))
    imwrite(os.path.join(cfg.SAVE_DIR, name+'_u16.tif'),
            (np.clip(arr01,0,1)*65535).astype(np.uint16))
    print(f"  ✓ {name}: range[{arr01.min():.3f},{arr01.max():.3f}] -> float + u16")

print("\n  Brightness konsisten: semua output skala [0,1] -> u16 penuh, tanpa denorm ke raw.")
print("  Selesai. Output di", cfg.SAVE_DIR)